In [1]:
import requests
import json

url = "https://servapibi.xm.com.co/daily"

payload = {
    "MetricId": "PorcVoluUtilDiar",
    "StartDate": "2026-10-01",
    "EndDate": "2026-10-07",
    "Entity": "Embalse"
}

respuesta = requests.post(url, json=payload, timeout=60)

print("HTTP:", respuesta.status_code)

datos = respuesta.json()

print(json.dumps(datos, indent=2, ensure_ascii=False)[:10000])

HTTP: 200
{
  "Metric": {
    "Id": "PorcVoluUtilDiar",
    "Name": "Volumen Útil  diario % por Embalse",
    "StartDate": "2026-10-01T00:00:00",
    "EndDate": "2026-10-07T00:00:00"
  },
  "Items": [
    {
      "Date": "2026-10-01",
      "DailyEntities": [
        {
          "Id": "Embalse",
          "Name": "AGREGADO BOGOTA",
          "Value": 0.66393
        }
      ]
    },
    {
      "Date": "2026-10-01",
      "DailyEntities": [
        {
          "Id": "Embalse",
          "Name": "ALTOANCHICAYA",
          "Value": 0.364
        }
      ]
    },
    {
      "Date": "2026-10-01",
      "DailyEntities": [
        {
          "Id": "Embalse",
          "Name": "AMANI",
          "Value": 0.61767
        }
      ]
    },
    {
      "Date": "2026-10-01",
      "DailyEntities": [
        {
          "Id": "Embalse",
          "Name": "BETANIA",
          "Value": 0.59686
        }
      ]
    },
    {
      "Date": "2026-10-01",
      "DailyEntities": [
        {
          "I

In [2]:
import pandas as pd

filas = []

for item in datos["Items"]:
    fecha = item["Date"]

    for entidad in item["DailyEntities"]:
        filas.append({
            "fecha": fecha,
            "embalse": entidad["Name"],
            "valor_original": entidad["Value"],
            "porcentaje": entidad["Value"] * 100
        })

df_embalses = pd.DataFrame(filas)

df_embalses.head(20)

,fecha,embalse,valor_original,porcentaje
0,2026-10-01,AGREGADO BOGOTA,0.66393,66.393
1,2026-10-01,ALTOANCHICAYA,0.36400,36.400
2,2026-10-01,AMANI,0.61767,61.767
3,2026-10-01,BETANIA,0.59686,59.686
4,2026-10-01,CALIMA1,0.62329,62.329
5,2026-10-01,CHUZA,0.92087,92.087
6,2026-10-01,EL QUIMBO,0.82438,82.438
7,2026-10-01,ESMERALDA,0.95922,95.922
8,2026-10-01,GUAVIO,0.88460,88.460
9,2026-10-01,ITUANGO,1.11174,111.174


In [3]:
df_embalses[
    df_embalses["fecha"] == "2026-10-01"
].sort_values("porcentaje").head(10)

,fecha,embalse,valor_original,porcentaje
16,2026-10-01,PRADO,0.35341,35.341
1,2026-10-01,ALTOANCHICAYA,0.36400,36.400
17,2026-10-01,PUNCHINA,0.40969,40.969
15,2026-10-01,PORCE III,0.48434,48.434
14,2026-10-01,PORCE II,0.53726,53.726
18,2026-10-01,RIOGRANDE2,0.54803,54.803
3,2026-10-01,BETANIA,0.59686,59.686
13,2026-10-01,PLAYAS,0.61433,61.433
23,2026-10-01,URRA1,0.61726,61.726
2,2026-10-01,AMANI,0.61767,61.767


In [4]:
menores_por_dia = (
    df_embalses
    .sort_values(["fecha", "porcentaje"])
    .groupby("fecha")
    .head(5)
    [["fecha", "embalse", "porcentaje"]]
)

menores_por_dia

,fecha,embalse,porcentaje
16,2026-10-01,PRADO,35.341
1,2026-10-01,ALTOANCHICAYA,36.400
17,2026-10-01,PUNCHINA,40.969
15,2026-10-01,PORCE III,48.434
14,2026-10-01,PORCE II,53.726
41,2026-10-02,PUNCHINA,27.980
40,2026-10-02,PRADO,36.060
25,2026-10-02,ALTOANCHICAYA,37.586
39,2026-10-02,PORCE III,51.430
38,2026-10-02,PORCE II,55.193


In [5]:
resumen_persistencia = (
    menores_por_dia
    .groupby("embalse")
    .agg(
        dias_entre_5_mas_bajos=("fecha", "count"),
        porcentaje_promedio=("porcentaje", "mean"),
        porcentaje_minimo=("porcentaje", "min"),
        porcentaje_maximo=("porcentaje", "max")
    )
    .sort_values(
        ["dias_entre_5_mas_bajos", "porcentaje_promedio"],
        ascending=[False, True]
    )
)

resumen_persistencia

,dias_entre_5_mas_bajos,porcentaje_promedio,porcentaje_minimo,porcentaje_maximo
embalse,,,,
PRADO,7,36.156000,35.341,36.628
ALTOANCHICAYA,6,39.566333,25.452,54.866
PUNCHINA,5,36.237600,13.626,55.619
PORCE II,5,50.032400,45.435,55.193
PLAYAS,4,46.361000,36.485,55.269
PORCE III,4,49.825750,41.690,57.749
ITUANGO,3,39.263667,37.967,40.586
RIOGRANDE2,1,56.285000,56.285,56.285
